In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [2]:
df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")

In [3]:
df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [4]:
df.info

<bound method DataFrame.info of             id Driver Compound                   Race  Year  PitStop  \
0            0   D109     HARD    Canadian Grand Prix  2022        0   
1            1   D086     HARD       Dutch Grand Prix  2025        1   
2            2    ZON     HARD    Austrian Grand Prix  2022        0   
3            3    SPE   MEDIUM     Pre-Season Testing  2023        0   
4            4   D019     HARD  Azerbaijan Grand Prix  2022        1   
...        ...    ...      ...                    ...   ...      ...   
439135  439135   D755   MEDIUM       Miami Grand Prix  2023        0   
439136  439136   D731   MEDIUM       Miami Grand Prix  2023        0   
439137  439137   D716   MEDIUM       Miami Grand Prix  2023        0   
439138  439138   D665     HARD   Abu Dhabi Grand Prix  2023        0   
439139  439139   D714     HARD       Miami Grand Prix  2023        0   

        LapNumber  Stint  TyreLife  Position  LapTime (s)  LapTime_Delta  \
0              50      2   

In [5]:
df.columns.to_list()

['id',
 'Driver',
 'Compound',
 'Race',
 'Year',
 'PitStop',
 'LapNumber',
 'Stint',
 'TyreLife',
 'Position',
 'LapTime (s)',
 'LapTime_Delta',
 'Cumulative_Degradation',
 'RaceProgress',
 'Position_Change',
 'PitNextLap']

In [6]:
df.isnull().sum()

id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64

In [7]:
# Option 1: select numeric columns first
num_df = df.select_dtypes(include=["int64", "float64"])
corr_matrix = num_df.corr()
corr_matrix

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
id,1.000000,0.002153,-0.000214,0.000852,0.002810,-0.000167,0.001732,0.003059,0.000116,-0.001201,0.000925,0.000015,-0.000097
Year,0.002153,1.000000,0.090767,0.008556,-0.055644,0.041472,0.039931,-0.047048,-0.011863,0.036667,-0.068874,0.008429,0.125267
PitStop,-0.000214,0.090767,1.000000,0.111431,0.143869,-0.134678,0.093536,-0.000437,-0.008458,-0.088052,0.059177,-0.107659,0.048567
LapNumber,0.000852,0.008556,0.111431,1.000000,0.724390,0.648408,0.001951,-0.128485,0.031275,-0.173135,0.964539,0.010347,0.267057
Stint,0.002810,-0.055644,0.143869,0.724390,1.000000,0.157733,0.151200,-0.068869,0.017193,-0.118171,0.710020,-0.014225,0.198193
TyreLife,-0.000167,0.041472,-0.134678,0.648408,0.157733,1.000000,-0.120744,-0.133304,0.035947,-0.111742,0.622660,0.052309,0.273510
Position,0.001732,0.039931,0.093536,0.001951,0.151200,-0.120744,1.000000,0.017899,-0.012633,-0.026649,0.006937,-0.316446,0.021348
LapTime (s),0.003059,-0.047048,-0.000437,-0.128485,-0.068869,-0.133304,0.017899,1.000000,0.138818,0.205705,-0.094317,-0.002352,-0.034096
LapTime_Delta,0.000116,-0.011863,-0.008458,0.031275,0.017193,0.035947,-0.012633,0.138818,1.000000,0.053713,0.036777,0.005837,-0.004946
Cumulative_Degradation,-0.001201,0.036667,-0.088052,-0.173135,-0.118171,-0.111742,-0.026649,0.205705,0.053713,1.000000,-0.137938,-0.000067,-0.167401


In [8]:
# If you want correlation with target only:
corr_with_target = num_df.corr()["PitNextLap"].sort_values(ascending=False)
print(corr_with_target)

PitNextLap                1.000000
TyreLife                  0.273510
LapNumber                 0.267057
Stint                     0.198193
RaceProgress              0.185477
Year                      0.125267
PitStop                   0.048567
Position_Change           0.046230
Position                  0.021348
id                       -0.000097
LapTime_Delta            -0.004946
LapTime (s)              -0.034096
Cumulative_Degradation   -0.167401
Name: PitNextLap, dtype: float64


In [9]:
new_df = df.copy()

In [10]:
X = new_df.drop(columns=["PitNextLap", "id"])
y = new_df["PitNextLap"]

In [11]:
X.columns


Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change'],
      dtype='object')

In [12]:
y.value_counts(normalize=True)

PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [14]:
!pip install catboost -q

In [15]:
cat_cols = ["Driver", "Compound", "Race"]
cat_features = [X_train.columns.get_loc(c) for c in cat_cols]
cat_features

[0, 1, 2]

In [16]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    depth=6,
    learning_rate=0.1,
    iterations=500,
    random_seed=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	test: 0.8994931	best: 0.8994931 (0)	total: 341ms	remaining: 2m 50s
100:	test: 0.9358063	best: 0.9358063 (100)	total: 21.7s	remaining: 1m 25s
200:	test: 0.9411612	best: 0.9411612 (200)	total: 42s	remaining: 1m 2s
300:	test: 0.9436711	best: 0.9436711 (300)	total: 1m 1s	remaining: 41s
400:	test: 0.9452373	best: 0.9452373 (400)	total: 1m 22s	remaining: 20.3s
499:	test: 0.9462246	best: 0.9462246 (499)	total: 1m 42s	remaining: 0us

bestTest = 0.946224615
bestIteration = 499



CatBoostClassifier(depth=6, eval_metric='AUC', iterations=500, learning_rate=0.1, loss_function='Logloss', random_seed=42, verbose=100)

In [17]:
from sklearn.metrics import roc_auc_score

# predicted probability for class 1
y_valid_pred_proba = model.predict_proba(X_valid)[:, 1]

auc = roc_auc_score(y_valid, y_valid_pred_proba)
print("Validation AUC:", auc)

Validation AUC: 0.9462246150450173


In [18]:
import pandas as pd

imp_feature = model.get_feature_importance()
fea_Imp_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": imp_feature
}).sort_values(by="importance", ascending=False)

fea_Imp_df

,feature,importance
3,Year,49.953615
6,Stint,13.883893
7,TyreLife,9.412075
10,LapTime_Delta,4.578811
2,Race,4.010247
12,RaceProgress,3.854085
1,Compound,3.829619
5,LapNumber,2.146926
13,Position_Change,1.744711
9,LapTime (s),1.619172


**TEST DF**

In [19]:
test_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/test.csv")
test_df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [20]:
test_X = test_df[X_train.columns]

In [21]:
test_pred_proba = model.predict_proba(test_X)[:, 1]

In [22]:
sample_sub = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv")
sample_sub.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0


In [23]:
submission = sample_sub.copy()
submission["PitNextLap"] = test_pred_proba
submission.head()

,id,PitNextLap
0,439140,0.004756
1,439141,0.004406
2,439142,0.005548
3,439143,0.067107
4,439144,0.787378


In [24]:
submission.to_csv("submission.csv", index=False)